In [2]:
import torch
from pathlib import Path
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from peft import PeftModel, PeftConfig

PROJECT_ROOT  = Path.home() / "icidea_llm_ids"
CHECKPOINTS   = PROJECT_ROOT / "checkpoints"

device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Device: {device}")

LABEL_MAP = {0: "NORMAL", 1: "DOS", 2: "FUZZY", 3: "GEAR", 4: "RPM"}
checkpoint_path = str(CHECKPOINTS / "securebert_can_final")

print("Loading SecureBERT + LoRA checkpoint...")

# Step 1: Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(checkpoint_path)

# Step 2: Load base model first
base_model = AutoModelForSequenceClassification.from_pretrained(
    "ehsanaghaei/SecureBERT",
    num_labels=5,
    ignore_mismatched_sizes=True,
)

# Step 3: Load LoRA adapter on top
model = PeftModel.from_pretrained(base_model, checkpoint_path)
model = model.to(device)
model.eval()

print(f"✓ Model loaded with LoRA adapter")
print(f"  Parameters: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

# Quick inference test
sample_text = "CAN Bus Telemetry Sequence (14 frames): [001] T=1478198377.185 ID=0000 DLC=8 DATA=00 00 00 00 00 00 00 00"
inputs = tokenizer(
    sample_text,
    return_tensors="pt",
    truncation=True,
    max_length=512,
    padding="max_length"
).to(device)

with torch.no_grad():
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=-1)

pred = probs.argmax().item()
print(f"\nTest inference:")
print(f"  Prediction: {LABEL_MAP[pred]} (confidence: {probs.max().item():.3f})")
print(f"  ✓ Model working correctly" if LABEL_MAP[pred] == "DOS"
      else f"  ⚠ Unexpected: {LABEL_MAP[pred]}")

Device: mps
Loading SecureBERT + LoRA checkpoint...


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at ehsanaghaei/SecureBERT and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✓ Model loaded with LoRA adapter
  Parameters: 125.5M

Test inference:
  Prediction: DOS (confidence: 0.986)
  ✓ Model working correctly


In [3]:
import shap
print(f"SHAP version: {shap.__version__}")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


SHAP version: 0.44.1
